In [1]:
import sys
# import matplotlib.pyplot as plt
import importlib
# sys.path.append('C:/Users/kanoutem/Desktop/Doctorat/Codes/feat_select/RSR_TS')
# sys.path.append('/home/mkanoute/Desktop/postdoc/RSR_TS')

# sys.path.append('C:/Users/kanoutem/Desktop/Doctorat/Codes/feat_select/RSR_TS/Clustering')
# sys.path.append('/home/mkanoute/Desktop/postdoc/RSR_TS/Clustering')

import numpy as np

from scipy.optimize import minimize
import torch
import pandas as pd
from sklearn.preprocessing import StandardScaler

from grad_fs_elem import *
import matplotlib.pyplot as plt

In [2]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score

# Synthetic data

$
f_1 \sim \mathcal{N}(1,5^2),
f_2 \sim \mathcal{N}(0,2^2),
f_3 \sim \mathcal{N}(2,7^2),
f_4 \sim \mathcal{N}(5,3^2),
f_5 \sim \mathcal{N}(0,1),$  \
$f_6 \sim \mathcal{N}(0,0.3^2),
f_7 \sim \mathcal{N}(3,2^2),
f_8 \sim \mathcal{N}(11,1), 
f_9 \sim \mathcal{N}(0.3,0.3^2),
f_{10} \sim \mathcal{N}(0,0.7^2).
.
$

 For regression, 3 continuous outputs are defined 
\begin{gather*} 
Y_1 =  cos(sin(f_8)) + \epsilon_{Y_1} ~\text{ where } ~ \epsilon_{Y_1} \sim \mathcal{N}(0, 0.1^2) \\ 
Y_2 =  f_1^2 sin(\dfrac{f_3}{f_1}) + \epsilon_{Y_2} ;  ~\text{ where } ~ \epsilon_{Y_2} \sim \mathcal{N}(0, 0.1^2) \\
Y_3 =  log(\dfrac{1}{f_1^2} + f_5^2 + 1) + \epsilon_{Y_3} ~\text{ where } ~ \epsilon_{Y_3} \sim \mathcal{N}(0, 0.1^2)
\end{gather*}

In [3]:
def generate_data(random_state, n_samples, output_consid = "all"):
    obj_data = {}
    np.random.seed(random_state)
    obj_data["f1"] = np.random.normal(1, 5, n_samples)
    obj_data["f2"] = np.random.normal(0, 2, n_samples)
    obj_data["f3"] = np.random.normal(2, 7, n_samples)
    obj_data["f4"] = np.random.normal(5, 3, n_samples)
    obj_data["f5"] = np.random.normal(0, 1, n_samples)
    obj_data["f6"] = np.random.normal(0, 0.3, n_samples)
    obj_data["f7"] = np.random.normal(3, 2, n_samples)
    obj_data["f8"] = np.random.normal(11, 1, n_samples)
    obj_data["f9"] = np.random.normal(0.3, 0.3, n_samples)
    obj_data["f10"] = np.random.normal(0, 0.7, n_samples)
    
    if output_consid == "Y1":
        obj_data["Y1"] = np.cos(np.sin(obj_data["f8"])) + np.random.normal(0, 0.1, n_samples)
    elif output_consid == "Y2":
        obj_data["Y2"] = (obj_data["f1"]**2)*np.sin(obj_data["f3"]/obj_data["f1"]) +  np.random.normal(0, 0.1, n_samples)
    elif output_consid == "Y3":
        obj_data["Y3"] = np.log1p(1/(obj_data["f1"]**2) + obj_data["f5"]**2) + np.random.normal(0, 0.1, n_samples)
    elif output_consid == "all":
        obj_data["Y1"] = np.cos(np.sin(obj_data["f8"])) + np.random.normal(0, 0.1, n_samples)
        obj_data["Y2"] = (obj_data["f1"]**2)*np.sin(obj_data["f3"]/obj_data["f1"]) +  np.random.normal(0, 0.1, n_samples)
        obj_data["Y3"] = np.log1p(1/(obj_data["f1"]**2) + obj_data["f5"]**2) + np.random.normal(0, 0.1, n_samples)
        
    obj_data_df = pd.DataFrame(obj_data)
    return obj_data_df, None

# Experiments

The first experiment is single target regression with  $Y_1$ as target \,

THe second experiment is the multi-target regression with $Y_1, Y_2, Y_3$ as targets  

In [4]:
random_state, n_samples = 5, 1000

## Experiment 1

In [5]:
data_df_mono, _ = generate_data(random_state, n_samples, output_consid = "Y1")
X0_mono, y0_mono = data_df_mono.iloc[:, :-1].values, data_df_mono.iloc[:, [-1]].values
sc_x_mono, sc_y_mono = StandardScaler(), StandardScaler()
X_mono, y_mono = sc_x_mono.fit_transform(X0_mono), sc_y_mono.fit_transform(y0_mono)

Feature selection step for the two experiments

In [6]:
activ_func, nb_neurons = "tanh", 400
params_optim = {"eps": 1e-8, "maxiter": 400, "maxfun": 15000, "maxls": 20, "maxcor": 100}
lbda, C  = 1e-1, 1e1
grad_FS_ELM_impl_mono = grad_FS_ELM(lbda, C, activ_func, nb_neurons, params_optim = params_optim)

In [7]:
alpha_min_mono = grad_FS_ELM_impl_mono.fit(X_mono, y_mono, type_optim = "grad_variant")

self.x0_user (10,)
0 tensor(320.8463, dtype=torch.float64) None
1 tensor(143.5995, dtype=torch.float64) 1.0
2 tensor(141.9709, dtype=torch.float64) 0.015625
3 tensor(131.5482, dtype=torch.float64) 0.03125
4 tensor(129.6367, dtype=torch.float64) 0.0078125


/home/mkanoute/Desktop/postdoc/RSR_TS/grad_fs_elem.py:74: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  alpha = torch.tensor(alpha0, dtype=torch.float64, device=self.device)


5 tensor(137.3205, dtype=torch.float64) 0.015625
6 tensor(155.4159, dtype=torch.float64) 0.0625
7 tensor(134.3063, dtype=torch.float64) 1.0
8 tensor(134.1860, dtype=torch.float64) 0.00390625
9 tensor(140.2579, dtype=torch.float64) 0.015625
10 tensor(118.9005, dtype=torch.float64) 0.015625
11 tensor(119.1897, dtype=torch.float64) 0.00048828125
Arrêt : variation relative de la fonction (ftol)


In [8]:
cols = np.array(list(data_df_mono)[:-y0_mono.shape[1]])
weights_var_mono = alpha_min_mono["x"]
weights_var_ord_ind_mono = np.argsort(-weights_var_mono)
Cols_ord_with_weights_mono = pd.DataFrame({"ranked vars": cols[weights_var_ord_ind_mono], "weights": weights_var_mono[weights_var_ord_ind_mono]})

features ranked by importance

In [9]:
Cols_ord_with_weights_mono

,ranked vars,weights
0,f8,0.970699
1,f6,0.515273
2,f9,0.502065
3,f4,0.402235
4,f7,0.245065
5,f10,0.149001
6,f2,0.068873
7,f5,0.014382
8,f1,0.000000
9,f3,0.000000


Prediction from the model obtained during the selection 

In [10]:
preds_sc_mono = grad_FS_ELM_impl_mono.predict(X_mono)
preds_mono = sc_y_mono.inverse_transform(preds_sc_mono)

MSE for the two experiments

In [11]:
mean_squared_error(y0_mono.ravel(), preds_mono.ravel())

0.006073254268096634

## Experiment 2

In [12]:
data_df_multi, _ = generate_data(random_state, n_samples, output_consid = "all")
X0_multi, y0_multi = data_df_multi.iloc[:, :-3].values, data_df_multi.iloc[:, -3:].values
sc_x_multi, sc_y_multi = StandardScaler(), StandardScaler()
X_multi, y_multi = sc_x_multi.fit_transform(X0_multi), sc_y_multi.fit_transform(y0_multi)

Feature selection step for the two experiments

In [13]:
activ_func, nb_neurons = "tanh", 400
params_optim = {"eps": 1e-8, "maxiter": 400, "maxfun": 15000, "maxls": 20, "maxcor": 100}
lbda, C  = 1e-2, 1e1
grad_FS_ELM_impl_multi = grad_FS_ELM(lbda, C, activ_func, nb_neurons, params_optim = params_optim)

In [14]:
alpha_min_multi = grad_FS_ELM_impl_multi.fit(X_multi, y_multi, type_optim = "grad_variant")

self.x0_user (10,)
0 tensor(714.7491, dtype=torch.float64) None
1 tensor(368.3045, dtype=torch.float64) 0.03125
2 tensor(306.5466, dtype=torch.float64) 0.00048828125
3 tensor(284.4461, dtype=torch.float64) 0.0078125
4 tensor(276.4207, dtype=torch.float64) 0.001953125


/home/mkanoute/Desktop/postdoc/RSR_TS/grad_fs_elem.py:74: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  alpha = torch.tensor(alpha0, dtype=torch.float64, device=self.device)


5 tensor(276.4041, dtype=torch.float64) 0.000244140625
6 tensor(278.0498, dtype=torch.float64) 0.00048828125
7 tensor(274.8449, dtype=torch.float64) 0.001953125
8 tensor(268.7226, dtype=torch.float64) 0.00048828125
9 tensor(268.0297, dtype=torch.float64) 0.001953125
10 tensor(266.3345, dtype=torch.float64) 0.0009765625
11 tensor(265.5277, dtype=torch.float64) 0.000244140625
12 tensor(264.6405, dtype=torch.float64) 0.00048828125
13 tensor(264.6297, dtype=torch.float64) 0.000244140625
14 tensor(264.1434, dtype=torch.float64) 0.000244140625
15 tensor(264.1860, dtype=torch.float64) 0.00048828125
Arrêt : variation relative de la fonction (ftol)


In [15]:
weights_var_multi = alpha_min_multi["x"]
weights_var_ord_ind_multi = np.argsort(-weights_var_multi)
Cols_ord_with_weights_multi = pd.DataFrame({"ranked vars": cols[weights_var_ord_ind_multi], "weights": weights_var_multi[weights_var_ord_ind_multi]})

features ranked by importance

In [16]:
Cols_ord_with_weights_multi

,ranked vars,weights
0,f1,1.000000
1,f8,0.663051
2,f3,0.401287
3,f5,0.327918
4,f9,0.120242
5,f7,0.082524
6,f6,0.031852
7,f2,0.005757
8,f4,0.000000
9,f10,0.000000


Prediction from the model obtained during the selection

In [17]:
preds_sc_multi = grad_FS_ELM_impl_multi.predict(X_multi)
preds_multi = sc_y_multi.inverse_transform(preds_sc_multi)

MSE for the two experiments

In [18]:
mean_squared_error(y0_multi.ravel(), preds_multi.ravel())

2.479671382817886